# SurfaceTarget-CRC — phase 05 — patch 5 3

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.3
#
# EVIDENCE-BASED CANDIDATE TRIAGE
#
# Objectives:
# 1. Validate Phase 5.2 classification results
# 2. Audit RNA evidence consistency
# 3. Identify candidates with annotated surface topology
# 4. Distinguish proteomic support from missing coverage
# 5. Preserve normal-tissue expression concerns
# 6. Preserve HPA staining reliability information
# 7. Separate tumor-cell and stromal hypotheses
# 8. Create research follow-up categories
# 9. Identify candidate evidence gaps
# 10. Save results and provenance
#
# IMPORTANT:
# - No therapeutic efficacy scores
# - No clinical safety scores
# - No candidate genes excluded
# - No arbitrary weighted ranking
# - DepMap dependency does not determine ADC eligibility
# - Missing evidence is not negative evidence
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import hashlib
import json


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"

RESULTS.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

SOURCE_PATH = (
    RESULTS /
    "surfacetarget_phase5_target_classification_v1.csv"
)

PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_classification_provenance.json"
)

OUTPUT_PATH = (
    RESULTS /
    "surfacetarget_phase5_evidence_triage_v1.csv"
)

LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.3")
print("EVIDENCE-BASED CANDIDATE TRIAGE")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE RESEARCH INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)

for path in [SOURCE_PATH, PROVENANCE_PATH]:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(PROVENANCE_PATH, "r") as handle:
    previous_provenance = json.load(handle)

data = pd.read_csv(
    SOURCE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)

if len(data) != 948:
    raise RuntimeError("Expected 948 candidates.")

if data["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate gene identifiers.")

required = {
    "gene_id",
    "gene_name",
    "baseline_log2fc",
    "baseline_padj",
    "sensitivity_log2fc",
    "sensitivity_padj",
    "tier_a_median_cpm",
    "tier_a_fraction_cpm_ge1",
    "paired_direction_supported",
    "protein_enriched",
    "mss_complete_pairs",
    "phase5_topology_class",
    "phase5_surface_review_group",
    "phase5_protein_evidence_state",
    "phase5_normal_ihc_available",
    "hpa_reliable_medium_high_tissues",
    "phase5_normal_colon_max_ihc_score",
    "phase5_normal_colon_reliable_ihc_score",
    "phase5_normal_kidney_max_ihc_score",
    "phase5_normal_kidney_reliable_ihc_score",
    "phase5_normal_skin_max_ihc_score",
    "phase5_normal_skin_reliable_ihc_score",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect"
}

missing = required - set(data.columns)

if missing:
    raise RuntimeError(
        f"Missing required fields: {sorted(missing)}"
    )

if int(
    previous_provenance.get("candidate_count", -1)
) != 948:

    raise RuntimeError(
        "Patch 5.2 provenance count mismatch."
    )

source_hash = sha256_file(SOURCE_PATH)

print("\nSource candidates:", len(data))
print("Source fields:", len(data.columns))
print("Source SHA256:", source_hash)
print("Input validation: PASSED")


# ------------------------------------------------------------
# 3. BOOLEAN AND NUMERIC HELPERS
# ------------------------------------------------------------

def as_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    valid = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(valid))
    )

    if unexpected.any():
        raise RuntimeError(
            "Unexpected boolean values: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


def as_numeric(column):

    return pd.to_numeric(
        data[column],
        errors="coerce"
    )


protein = as_bool(data["protein_enriched"])
paired = as_bool(data["paired_direction_supported"])
ihc = as_bool(data["phase5_normal_ihc_available"])

if protein.sum() != 20:
    raise RuntimeError("Expected 20 protein-supported genes.")

if ihc.sum() != 690:
    raise RuntimeError("Expected 690 IHC-annotated genes.")


# ------------------------------------------------------------
# 4. AUDIT RNA EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING TRANSCRIPTOMIC SUPPORT")
print("-" * 65)

baseline_lfc = as_numeric("baseline_log2fc")
baseline_padj = as_numeric("baseline_padj")

sensitivity_lfc = as_numeric("sensitivity_log2fc")
sensitivity_padj = as_numeric("sensitivity_padj")

tier_a_cpm = as_numeric("tier_a_median_cpm")
tier_a_fraction = as_numeric(
    "tier_a_fraction_cpm_ge1"
)

# These thresholds are exploratory evidence
# descriptors, not therapeutic eligibility rules.

baseline_support = (
    baseline_lfc > 1
) & (
    baseline_padj < 0.05
)

sensitivity_support = (
    sensitivity_lfc > 1
) & (
    sensitivity_padj < 0.05
)

tier_a_expression_support = (
    tier_a_cpm >= 1
) & (
    tier_a_fraction >= 0.5
)

# Matched-patient directional support is
# supplementary and should not be described
# as independent external validation.

data["phase5_rna_baseline_supported"] = (
    baseline_support.fillna(False)
)

data["phase5_rna_sensitivity_supported"] = (
    sensitivity_support.fillna(False)
)

data["phase5_rna_tier_a_expression_supported"] = (
    tier_a_expression_support.fillna(False)
)

data["phase5_rna_paired_direction_supported"] = paired

data["phase5_rna_evidence_pattern"] = np.select(

    [
        (
            baseline_support
            & sensitivity_support
            & tier_a_expression_support
            & paired
        ).to_numpy(dtype=bool),

        (
            baseline_support
            & sensitivity_support
            & tier_a_expression_support
        ).to_numpy(dtype=bool),

        (
            baseline_support
            & sensitivity_support
        ).to_numpy(dtype=bool)
    ],

    [
        "Reproducible DE + Tier A expression + paired direction",
        "Reproducible DE + Tier A expression",
        "Reproducible DE; additional support requires review"
    ],

    default="Partial or inconsistent RNA evidence"
)

print("\nRNA evidence patterns:")

print(
    data["phase5_rna_evidence_pattern"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 5. DEFINE SURFACE-TOPOLOGY EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING SURFACE-TOPOLOGY EVIDENCE")
print("-" * 65)

STRONGER_TOPOLOGY = {
    "Membrane with annotated extracellular region",
    "GPI-anchored membrane-associated"
}

UNRESOLVED_MEMBRANE = {
    "Membrane-associated; extracellular topology unresolved",
    "Transmembrane; surface localization unresolved"
}

EXTRACELLULAR_NONMEMBRANE = {
    "Secreted/extracellular; membrane attachment not established"
}

topology = data["phase5_topology_class"]

stronger_surface = topology.isin(
    STRONGER_TOPOLOGY
)

unresolved_surface = topology.isin(
    UNRESOLVED_MEMBRANE
)

extracellular = topology.isin(
    EXTRACELLULAR_NONMEMBRANE
)

data["phase5_stronger_surface_annotation"] = (
    stronger_surface
)

data["phase5_unresolved_membrane_annotation"] = (
    unresolved_surface
)

data["phase5_extracellular_nonmembrane_annotation"] = (
    extracellular
)

print(
    "\nStronger surface-related annotations:",
    int(stronger_surface.sum())
)

print(
    "Membrane-related annotations requiring verification:",
    int(unresolved_surface.sum())
)

print(
    "Secreted/extracellular non-membrane annotations:",
    int(extracellular.sum())
)

print(
    "Other or insufficient topology evidence:",
    int(
        (
            ~stronger_surface
            & ~unresolved_surface
            & ~extracellular
        ).sum()
    )
)


# ------------------------------------------------------------
# 6. CLASSIFY EVIDENCE PACKAGES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING CROSS-OMICS EVIDENCE PACKAGES")
print("-" * 65)

# These categories represent available evidence.
# They do not measure therapeutic efficacy.

conditions = [

    (
        stronger_surface & protein
    ).to_numpy(dtype=bool),

    (
        stronger_surface & ~protein
    ).to_numpy(dtype=bool),

    (
        unresolved_surface & protein
    ).to_numpy(dtype=bool),

    (
        unresolved_surface & ~protein
    ).to_numpy(dtype=bool),

    (
        extracellular & protein
    ).to_numpy(dtype=bool),

    (
        extracellular & ~protein
    ).to_numpy(dtype=bool)
]

choices = [
    "RNA + qualifying protein + stronger surface annotation",

    "RNA + stronger surface annotation; protein support incomplete or nonqualifying",

    "RNA + qualifying protein; membrane accessibility unresolved",

    "RNA + unresolved membrane evidence; protein support incomplete or nonqualifying",

    "RNA + qualifying protein; extracellular non-membrane hypothesis",

    "RNA + extracellular non-membrane annotation"
]

data["phase5_evidence_package"] = np.select(
    conditions,
    choices,
    default=(
        "RNA evidence; no qualifying surface topology "
        "in current annotations"
    )
)

print("\nEvidence-package distribution:")

print(
    data["phase5_evidence_package"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 7. DEFINE TARGETING REVIEW LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ASSIGNING BIOLOGICAL FOLLOW-UP LANES")
print("-" * 65)

# FAP is handled separately because its
# stromal biology requires a different
# validation strategy.
#
# Other proteins are not assumed to be
# tumor-cell-specific.

fap = data["gene_name"].eq("FAP")

data["phase5_followup_lane"] = np.select(

    [
        fap.to_numpy(dtype=bool),

        (
            stronger_surface & ~fap
        ).to_numpy(dtype=bool),

        unresolved_surface.to_numpy(dtype=bool),

        extracellular.to_numpy(dtype=bool)
    ],

    [
        "Stromal-associated targeting review",

        "Putative cell-surface antigen review",

        "Membrane localization/accessibility verification",

        "Extracellular microenvironment review"
    ],

    default=(
        "Topology clarification or alternative modality review"
    )
)

print("\nFollow-up lane distribution:")

print(
    data["phase5_followup_lane"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 8. AUDIT NORMAL-TISSUE EXPRESSION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING NORMAL-TISSUE SELECTIVITY QUESTIONS")
print("-" * 65)

FOCUS_TISSUES = [
    "colon",
    "kidney",
    "skin"
]

reported_medium_high = pd.Series(
    False,
    index=data.index
)

reliably_reported_medium_high = pd.Series(
    False,
    index=data.index
)

for tissue in FOCUS_TISSUES:

    observed = as_numeric(
        f"phase5_normal_{tissue}_max_ihc_score"
    )

    reliable = as_numeric(
        f"phase5_normal_{tissue}_reliable_ihc_score"
    )

    reported_medium_high |= (
        observed >= 2
    ).fillna(False)

    reliably_reported_medium_high |= (
        reliable >= 2
    ).fillna(False)


data["phase5_focus_normal_medium_high"] = (
    reported_medium_high
)

data["phase5_focus_normal_reliable_medium_high"] = (
    reliably_reported_medium_high
)

reliable_tissues = as_numeric(
    "hpa_reliable_medium_high_tissues"
)

# Evidence-review categories, not toxicity levels.

data["phase5_normal_tissue_review_state"] = np.select(

    [
        (~ihc).to_numpy(dtype=bool),

        reliably_reported_medium_high.to_numpy(dtype=bool),

        reported_medium_high.to_numpy(dtype=bool),

        (
            ihc
            & (reliable_tissues > 0).fillna(False)
        ).to_numpy(dtype=bool)
    ],

    [
        "Normal-tissue IHC unavailable",

        "Medium/high staining in focus organ with stronger IHC reliability",

        "Medium/high staining in focus organ; reliability requires review",

        "Reliable medium/high staining reported in other normal tissues"
    ],

    default=(
        "No medium/high focus-organ signal in available summary; "
        "not evidence of safety"
    )
)

print("\nNormal-tissue review states:")

print(
    data["phase5_normal_tissue_review_state"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 9. PRESERVE DEPMAP FUNCTIONAL EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING FUNCTIONAL DEPENDENCY CONTEXT")
print("-" * 65)

dependency = as_numeric(
    "depmap_crc_median_gene_effect"
)

depmap_coverage = as_numeric(
    "depmap_crc_models_measured"
)

data["phase5_depmap_functional_review"] = np.select(

    [
        (
            depmap_coverage == 0
        ).fillna(False).to_numpy(dtype=bool),

        (
            depmap_coverage >= 65
        ).fillna(False).to_numpy(dtype=bool)
    ],

    [
        "No colorectal CRISPR measurements",

        "High-coverage colorectal CRISPR observations"
    ],

    default="Partial colorectal CRISPR coverage"
)

print("\nFunctional evidence coverage:")

print(
    data["phase5_depmap_functional_review"]
    .value_counts()
    .to_string()
)

print("\nImportant:")

print(
    "CRISPR dependency is recorded but is "
    "not used to determine antibody-target eligibility."
)


# ------------------------------------------------------------
# 10. GENERATE FOLLOW-UP QUESTIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING BIOLOGICAL REVIEW QUESTIONS")
print("-" * 65)


def followup_questions(row):

    questions = []

    lane = row["phase5_followup_lane"]

    protein_state = str(
        row["phase5_protein_evidence_state"]
    )

    normal_state = str(
        row["phase5_normal_tissue_review_state"]
    )

    if lane == "Stromal-associated targeting review":

        questions.append(
            "Confirm fibroblast versus malignant-cell expression"
        )

    elif lane == "Putative cell-surface antigen review":

        questions.append(
            "Validate malignant-cell surface expression and epitope access"
        )

    elif lane == (
        "Membrane localization/accessibility verification"
    ):

        questions.append(
            "Confirm plasma-membrane residency and extracellular accessibility"
        )

    elif lane == "Extracellular microenvironment review":

        questions.append(
            "Investigate extracellular localization and targeting mechanism"
        )

    else:

        questions.append(
            "Resolve protein topology before antibody-target ranking"
        )


    if protein_state != (
        "Qualifying MSS paired-protein enrichment"
    ):

        questions.append(
            "Review protein measurement coverage and tumor enrichment"
        )


    if normal_state == "Normal-tissue IHC unavailable":

        questions.append(
            "Acquire independent normal-tissue expression evidence"
        )

    elif (
        "Medium/high staining" in normal_state
        or "Reliable medium/high" in normal_state
    ):

        questions.append(
            "Review normal-tissue cell types and staining reliability"
        )


    if row["gene_name"] == "TDGF1":

        questions.append(
            "Evaluate CRIPTO surface presentation and normal-tissue selectivity"
        )


    return "; ".join(questions)


data["phase5_followup_questions"] = (
    data.apply(
        followup_questions,
        axis=1
    )
)


# ------------------------------------------------------------
# 11. REVIEW SURFACE-RELATED CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SURFACE-RELATED CANDIDATE REVIEW")
print("-" * 65)

surface_review = data[
    data["phase5_followup_lane"].isin([
        "Putative cell-surface antigen review",
        "Stromal-associated targeting review"
    ])
].copy()

print(
    "\nCandidates with stronger surface-related annotations:",
    len(surface_review)
)

surface_protein = surface_review[
    as_bool(surface_review["protein_enriched"])
].copy()

print(
    "With qualifying CPTAC protein enrichment:",
    len(surface_protein)
)

print("\nCross-omics-supported surface-related candidates:")

print(
    surface_protein[
        [
            "gene_name",
            "phase5_followup_lane",
            "phase5_topology_class",
            "phase5_protein_evidence_state",
            "phase5_normal_tissue_review_state",
            "depmap_crc_median_gene_effect"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. REVIEW ALL 20 PROTEIN-SUPPORTED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COMPLETE CROSS-OMICS CANDIDATE REVIEW")
print("-" * 65)

protein_review = data[
    protein
].copy()

if len(protein_review) != 20:
    raise RuntimeError(
        "Cross-omics candidate count changed."
    )

print("\n")

print(
    protein_review[
        [
            "gene_name",
            "phase5_followup_lane",
            "phase5_evidence_package"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 13. REVIEW LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE EVIDENCE REVIEW")
print("-" * 65)

leading = data[
    data["gene_name"].isin(
        LEADING_GENES
    )
].copy()

leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: i
        for i, gene in enumerate(
            LEADING_GENES
        )
    })
)

leading = leading.sort_values(
    "display_order"
)

print("\n")

print(
    leading[
        [
            "gene_name",
            "phase5_rna_evidence_pattern",
            "phase5_followup_lane",
            "phase5_protein_evidence_state",
            "phase5_normal_tissue_review_state"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 14. VERIFY SOURCE DATA WERE NOT MODIFIED
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VERIFYING DATA PRESERVATION")
print("-" * 65)

if len(data) != 948:
    raise RuntimeError(
        "Candidate count changed."
    )

if data["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate genes after triage."
    )

if int(
    as_bool(data["protein_enriched"]).sum()
) != 20:

    raise RuntimeError(
        "CPTAC support count changed."
    )

if int(
    as_bool(
        data["phase5_normal_ihc_available"]
    ).sum()
) != 690:

    raise RuntimeError(
        "Normal-IHC count changed."
    )

print("\nCandidate preservation: PASSED")
print("CPTAC support count: PASSED")
print("Normal-IHC evidence count: PASSED")


# ------------------------------------------------------------
# 15. PREPARE SUMMARY TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING EVIDENCE TRIAGE SUMMARY")
print("-" * 65)

data["_protein_supported_temp"] = protein.to_numpy()

lane_summary = (
    data.groupby(
        "phase5_followup_lane",
        dropna=False
    )
    .agg(
        candidates=("gene_id", "size"),

        cptac_supported=(
            "_protein_supported_temp",
            "sum"
        ),

        stronger_surface=(
            "phase5_stronger_surface_annotation",
            "sum"
        ),

        normal_ihc_available=(
            "phase5_normal_ihc_available",
            lambda x: int(as_bool(x).sum())
        )
    )
    .reset_index()
)

data = data.drop(
    columns=["_protein_supported_temp"]
)

print("\n")

print(
    lane_summary.to_string(index=False)
)

if int(lane_summary["candidates"].sum()) != 948:
    raise RuntimeError(
        "Triage summary lost candidates."
    )


# ------------------------------------------------------------
# 16. SAVE TRIAGE RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING EVIDENCE TRIAGE RESULTS")
print("-" * 65)

data.to_csv(
    OUTPUT_PATH,
    index=False
)

lane_summary.to_csv(
    RESULTS /
    "surfacetarget_phase5_followup_lane_summary.csv",
    index=False
)

surface_review.to_csv(
    RESULTS /
    "surfacetarget_phase5_surface_candidate_review.csv",
    index=False
)

surface_protein.to_csv(
    RESULTS /
    "surfacetarget_phase5_surface_crossomics_review.csv",
    index=False
)

protein_review.to_csv(
    RESULTS /
    "surfacetarget_phase5_protein_supported_review.csv",
    index=False
)

leading.to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_evidence_triage.csv",
    index=False
)


reloaded = pd.read_csv(
    OUTPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)

if len(reloaded) != 948:
    raise RuntimeError(
        "Saved triage table has incorrect dimensions."
    )

if reloaded["gene_id"].tolist() != data["gene_id"].tolist():
    raise RuntimeError(
        "Saved gene order changed."
    )

print("\nAll evidence triage tables saved.")
print("Saved table validation: PASSED")


# ------------------------------------------------------------
# 17. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING TRIAGE PROVENANCE")
print("-" * 65)

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": str(SOURCE_PATH),

    "source_sha256": source_hash,

    "output": str(OUTPUT_PATH),

    "output_sha256": sha256_file(OUTPUT_PATH),

    "candidate_count": 948,

    "protein_supported_count": 20,

    "surface_annotation_policy": (
        "Annotated extracellular membrane regions "
        "and GPI-associated membrane proteins "
        "form the stronger surface review group"
    ),

    "rna_policy": (
        "Exploratory baseline and sensitivity DE, "
        "Tier A expression and patient-paired "
        "directional support retained separately"
    ),

    "protein_policy": (
        "Preserve qualifying enrichment and "
        "nonqualifying or unavailable measurements"
    ),

    "normal_tissue_policy": (
        "Normal colon, kidney and skin staining "
        "flagged for biological and reliability review"
    ),

    "dependency_policy": (
        "CRISPR effects retained as descriptive "
        "functional evidence, not an ADC requirement"
    ),

    "stromal_policy": (
        "FAP assigned a separate stromal-associated "
        "targeting review lane"
    ),

    "candidate_exclusions": 0,

    "weighted_scoring_performed": False,

    "final_target_ranking_performed": False,

    "limitations": [

        "Surface-related annotations do not "
        "demonstrate antibody accessibility.",

        "Bulk tumor RNA and protein enrichment "
        "do not establish malignant-cell origin.",

        "CPTAC protein missingness is not "
        "interpreted as protein absence.",

        "HPA normal-tissue staining is not "
        "a clinical toxicity measurement.",

        "Normal-tissue review categories "
        "are not clinical safety categories.",

        "TCGA discovery Tier B is computationally "
        "low MSI, not independently confirmed MSS.",

        "DepMap colorectal models are not "
        "confirmed MSS by this analysis.",

        "No experimental target validation "
        "has been performed."

    ]
}

with open(
    META /
    "surfacetarget_phase5_evidence_triage_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )

print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.3 SUMMARY")
print("=" * 65)

print("\nTotal candidates:", len(data))

print(
    "Stronger surface-related annotations:",
    int(stronger_surface.sum())
)

print(
    "Cross-omics-supported candidates:",
    int(protein.sum())
)

print(
    "Cross-omics-supported, stronger "
    "surface-related candidates:",
    len(surface_protein)
)

print(
    "Biological follow-up lanes:",
    len(lane_summary)
)

print("\nSaved files:")

print(
    "1. surfacetarget_phase5_evidence_triage_v1.csv"
)

print(
    "2. surfacetarget_phase5_followup_lane_summary.csv"
)

print(
    "3. surfacetarget_phase5_surface_candidate_review.csv"
)

print(
    "4. surfacetarget_phase5_surface_crossomics_review.csv"
)

print(
    "5. surfacetarget_phase5_protein_supported_review.csv"
)

print(
    "6. surfacetarget_phase5_leading_evidence_triage.csv"
)

print(
    "7. surfacetarget_phase5_evidence_triage_provenance.json"
)

print("\nSource evidence: PRESERVED")
print("Candidate count: VALIDATED")
print("No candidates excluded.")
print("No therapeutic efficacy scores assigned.")
print("No final target ranking performed.")

print("\nPATCH 5.3 COMPLETE")
print("=" * 65)